In [1]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
import os
import logging
import time
from dotenv import load_dotenv

/Users/hachikaruanyakwee/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
#Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [3]:
#Load API credentials and other variables from .env file
load_dotenv()

max_comments = int(os.getenv("MAX_COMMENTS", "1000").strip())  
request_delay = int(os.getenv("REQUEST_DELAY", "2").strip()) 

In [4]:
# Read YouTube URLs from .env file
video_urls = [os.getenv(f"YOUTUBE_URL_{i}") for i in range(1, 100) if os.getenv(f"YOUTUBE_URL_{i}")]

if not video_urls:
    logging.warning("No YouTube URLs found in the .env file.")
    exit()

logging.info(f"Found the following YouTube URLs to process: {video_urls}")

data = []
downloader = YoutubeCommentDownloader()

for url in video_urls:
    logging.info(f"Fetching comments for URL: {url}")
    try:
        comments = downloader.get_comments_from_url(url, sort_by=0)

        for comment in comments:
            if len(data) >= max_comments:
                break

            timestamp = comment.get("time_text", "N/A")
            text = comment.get("text", "No comment text available")

            data.append([timestamp, text, "YouTube"])

        if len(data) >= max_comments:
            logging.info(f"Reached maximum number of comments: {max_comments}. Stopping.")
            break

        time.sleep(request_delay)

    except RuntimeError as e:
        logging.error(f"RuntimeError for URL {url}: {e}")
    except Exception as e:
        logging.error(f"An error occurred for URL {url}: {e}")

2025-03-17 12:36:36,136 - INFO - Found the following YouTube URLs to process: ['https://www.youtube.com/watch?v=nOOHIWAvvSQ', 'https://www.youtube.com/watch?v=eWa54HPP6ew', 'https://www.youtube.com/watch?v=TvBzIBcXQ3Q']
2025-03-17 12:36:36,137 - INFO - Fetching comments for URL: https://www.youtube.com/watch?v=nOOHIWAvvSQ
2025-03-17 12:37:21,005 - INFO - Reached maximum number of comments: 1000. Stopping.


In [5]:
#Save to CSV
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

file_path = os.path.join(data_dir, "youtubecomments_data.csv")

if len(data) > 0:
    df = pd.DataFrame(data, columns=["timestamp", "comment", "source"])
    try:
        df.to_csv(file_path, index=False)
        logging.info(f"✅ Successfully saved {len(df)} YouTube comments to: {file_path}")
        print(f"✅ Successfully collected {len(df)} YouTube comments! Saved to: {file_path}")
    except Exception as e:
        logging.error(f"❌ Error saving data to CSV: {e}")
        print(f"❌ Error saving data to CSV: {e}")
else:
    logging.warning("⚠️ No comments collected. Skipping CSV save.")
    print("⚠️ No comments collected. Nothing to save.")

print(f"📊 Total comments collected: {len(data)}")

2025-03-17 12:37:21,021 - INFO - ✅ Successfully saved 1000 YouTube comments to: ../data/youtubecomments_data.csv


✅ Successfully collected 1000 YouTube comments! Saved to: ../data/youtubecomments_data.csv
📊 Total comments collected: 1000
